In [7]:
# since we pulled new data we wanna repeat the same process we did from dropping columns, merging dataframes, feature engineeering columns
import pandas as pd
import os
print(os.getcwd())

/Users/hussaintaheri/Desktop/sports-win-predictor/notebooks


In [8]:
df1 = pd.read_csv("../data/nba_pbp_2019_2020.csv", sep = "w")
df2 = pd.read_csv("../data/nba_pbp_2018_2019.csv", sep = "w")
df3 = pd.read_csv("../data/nba_pbp_2017_2018.csv", sep = "w")
df4 = pd.read_csv("../data/nba_pbp_2016_2017.csv", sep = "w")
df5 = pd.read_csv("../data/nba_pbp_2015_2016.csv", sep = "w")
df6 = pd.read_csv("../data/nba_pbp_2014_2015.csv", sep = "w")

df1.shape, df2.shape, df3.shape, df4.shape, df5.shape, df6.shape

((506335, 25),
 (577400, 25),
 (590491, 25),
 (564652, 25),
 (597954, 25),
 (531467, 25))

In [9]:
dfs = [df1,df2, df3, df4, df5, df6]
df = pd.concat(dfs)
df.shape

(3368299, 25)

In [10]:
df[["scoreHome","scoreAway"]] = df.groupby(["gameId"])[["scoreHome", "scoreAway"]].ffill()

temp_df = df.groupby("gameId").last().reset_index()
temp_df["home_team_won"] = (temp_df["scoreHome"] > temp_df["scoreAway"]).astype(int)
df = pd.merge(df, temp_df[["home_team_won", "gameId"]], on = "gameId")

def parse_clock(clock, period):
    clock = clock.split("M")
    clock[0] = clock[0][2:]
    clock[1] = clock[1][:-4]

    clock_seconds_left = int(clock[0]) * 60 + int(clock[1])

    if period < 5:
        seconds_before_period = (period - 1) * 720
        seconds = seconds_before_period + (720 - clock_seconds_left)
    else:
        seconds_before_period = 2880 + (period - 5) * 300
        seconds = seconds_before_period + (300 - clock_seconds_left)

    return seconds

df["seconds"] = df.apply(lambda row: parse_clock(row["clock"], row["period"]), axis = 1)

df = pd.get_dummies(df, columns = ["actionType"], dtype = int)

df["location"] = df["location"].map({"h": 2, "v": 1})
df["location"] = df["location"].fillna(0)

,period,scoreHome,scoreAway,pointsTotal,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,1,0.0,0.0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,0.0,0.0,0,2.0,0,4,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1,0.0,0.0,0,1.0,0,12,0,0,0,0,0,0,1,0,0,0,0,0,0
3,1,0.0,0.0,0,2.0,0,15,0,0,0,0,0,0,0,1,0,0,0,0,0
4,1,0.0,0.0,0,2.0,0,21,0,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3368294,4,78.0,104.0,0,2.0,0,2841,0,0,0,0,0,0,0,1,0,0,0,0,0
3368295,4,78.0,104.0,0,2.0,0,2858,0,0,0,0,0,0,1,0,0,0,0,0,0
3368296,4,78.0,104.0,0,1.0,0,2859,0,0,0,0,0,0,0,1,0,0,0,0,0
3368297,4,78.0,106.0,184,1.0,0,2862,0,0,0,0,0,1,0,0,0,0,0,0,0


In [14]:
df = df.drop(columns = ['Unnamed: 0', 'gameId', 'actionNumber', 'clock', 'teamId', 'teamTricode', 'personId', 'playerName',
       'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult','isFieldGoal', 'description', 'subType', 'videoAvailable', 
                   'shotValue', 'actionId'])

In [16]:
df = df.drop("pointsTotal", axis = 1)

In [21]:
df.head(20)

,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,0.0,0.0,2.0,0,4,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1,0.0,0.0,1.0,0,12,0,0,0,0,0,0,1,0,0,0,0,0,0
3,1,0.0,0.0,2.0,0,15,0,0,0,0,0,0,0,1,0,0,0,0,0
4,1,0.0,0.0,2.0,0,21,0,0,0,0,0,0,1,0,0,0,0,0,0
5,1,0.0,0.0,2.0,0,24,0,0,0,0,0,0,0,1,0,0,0,0,0
6,1,0.0,0.0,2.0,0,30,0,0,0,0,0,0,1,0,0,0,0,0,0
7,1,0.0,0.0,1.0,0,34,0,0,0,0,0,0,0,1,0,0,0,0,0
8,1,0.0,0.0,1.0,0,39,0,0,0,0,0,0,1,0,0,0,0,0,0
9,1,0.0,0.0,2.0,0,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [22]:
df.shape

(3368299, 19)

In [23]:
df.dtypes

period                         int64
scoreHome                    float64
scoreAway                    float64
location                     float64
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object

In [24]:
df.isna().sum()

period                       0
scoreHome                    1
scoreAway                    1
location                     0
home_team_won                0
seconds                      0
actionType_Ejection          0
actionType_Foul              0
actionType_Free Throw        0
actionType_Instant Replay    0
actionType_Jump Ball         0
actionType_Made Shot         0
actionType_Missed Shot       0
actionType_Rebound           0
actionType_Substitution      0
actionType_Timeout           0
actionType_Turnover          0
actionType_Violation         0
actionType_period            0
dtype: int64

In [26]:
df[df["scoreHome"].isna()]

,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
1303379,1,NaN,NaN,1.0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [28]:
df = df.drop(1303379)
df.isna().sum()

period                       0
scoreHome                    0
scoreAway                    0
location                     0
home_team_won                0
seconds                      0
actionType_Ejection          0
actionType_Foul              0
actionType_Free Throw        0
actionType_Instant Replay    0
actionType_Jump Ball         0
actionType_Made Shot         0
actionType_Missed Shot       0
actionType_Rebound           0
actionType_Substitution      0
actionType_Timeout           0
actionType_Turnover          0
actionType_Violation         0
actionType_period            0
dtype: int64

In [29]:
df.head(20)

,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,0.0,0.0,2.0,0,4,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1,0.0,0.0,1.0,0,12,0,0,0,0,0,0,1,0,0,0,0,0,0
3,1,0.0,0.0,2.0,0,15,0,0,0,0,0,0,0,1,0,0,0,0,0
4,1,0.0,0.0,2.0,0,21,0,0,0,0,0,0,1,0,0,0,0,0,0
5,1,0.0,0.0,2.0,0,24,0,0,0,0,0,0,0,1,0,0,0,0,0
6,1,0.0,0.0,2.0,0,30,0,0,0,0,0,0,1,0,0,0,0,0,0
7,1,0.0,0.0,1.0,0,34,0,0,0,0,0,0,0,1,0,0,0,0,0
8,1,0.0,0.0,1.0,0,39,0,0,0,0,0,0,1,0,0,0,0,0,0
9,1,0.0,0.0,2.0,0,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [31]:
df.to_csv("../data/nba_pbp_2014_2020_fe.csv")